# PrimeKV — Quick Test Notebook

Run these cells top-to-bottom to install, test, and interactively
compare KV cache strategies. Works on CPU (free tier) or GPU.

**From your phone:** just tap each cell and hit the play button.

## 1. Clone and install

In [ ]:
!rm -rf /content/PrimeKV
!git clone https://github.com/arunvenkatadri/PrimeKV.git
%cd /content/PrimeKV
!git checkout claude/scaffold-primekv-Q9QKN
!pip install -e ".[dev,web]" -q

## 2. Run unit tests (no network, no GPU, ~3 seconds)

In [ ]:
!pytest tests/ -v

## 3. Run the comparison CLI with real GPT-2

Downloads GPT-2 (124M) on first run (~500 MB). Takes 30-60s on CPU.

In [ ]:
!python benchmarks/compare.py --model gpt2 --decode-tokens 16 --max-length 64

## 4. Run comparison from Python (more control)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from primekv.eval import Workload, run_comparison
from primekv.cache import PrimeKVCache
from primekv.classifier import RuleBasedClassifier, Tier
from primekv.baselines import FullCache, H2OCache, UniformQuantCache

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

num_layers = model.config.n_layer

caches = {
    "full":        FullCache(num_layers),
    "uniform_int4": UniformQuantCache(num_layers, bits=4),
    "h2o":         H2OCache(num_layers, capacity=32),
    "primekv":     PrimeKVCache(
        num_layers=num_layers,
        classifier=RuleBasedClassifier(anchor_prefix_len=4, semantic_stride=3),
        max_entries_per_tier={Tier.SUPPORTING: 32},
    ),
}

workload = Workload(
    prompt="System: you are a helpful assistant. User: What is the capital of France? Assistant:",
    decode_tokens=16,
    max_length=64,
)

report = run_comparison(caches, workload, model, tok)
print(report.to_markdown())
print()
for r in report.results:
    if r.generated:
        print(f"--- {r.name} ---")
        print(r.generated)
        print()

## 5. Launch interactive Gradio UI

This creates a **public share link** you can open in any browser tab
(or send to a collaborator). The link is active as long as this cell
is running.

In [ ]:
%cd /content/PrimeKV
from webui.app import build_demo

demo = build_demo()
demo.launch(share=True)

## 6. Inspect PrimeKV tier distribution

In [ ]:
from primekv.metrics import tier_distribution, summarize_stats

# Use the primekv cache from step 4 (still in memory)
pkv = caches["primekv"]

print("Tier distribution:")
for tier, count in tier_distribution(pkv).items():
    print(f"  {tier:12s}  {count} tokens")

print("\nCache stats:")
for k, v in summarize_stats(pkv.stats).items():
    print(f"  {str(k):20s}  {v}")

## 7. Real-scale validation (GPU required)

This section runs the 2D eviction × quantization sweep on a **real instruction-tuned model** (Phi-2, 2.7B) with a long prompt (~1500 tokens). This is what goes into the paper.

**Before running this section:** go to `Runtime → Change runtime type → T4 GPU` (or A100 if you have Colab Pro). Then restart the runtime and run cell 1 again to reinstall.

Expected runtime: ~5-15 minutes on T4, ~2-5 minutes on A100.

In [ ]:
%cd /content/PrimeKV
import torch

# Verify GPU is available.
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, "
    "then restart the runtime and re-run cells 1 and 7.1."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 7.2 Load a real model

**Qwen2.5-3B-Instruct** is instruction-tuned, open-weight, first-class in transformers (no `trust_remote_code`), and doesn't require HuggingFace authentication. It's a genuine step up from GPT-2.

On an A100 it runs in FP16 with room to spare. If you're on a T4 and this is too big, downgrade to `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
%cd /content/PrimeKV
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Qwen2.5-3B-Instruct is first-class in transformers (no trust_remote_code
# needed), instruction-tuned, open-weights (no HF auth), and ~6GB in FP16
# so it fits comfortably on an A100 or T4.
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cuda",
    attn_implementation="eager",   # needed for standard past_key_values
)
model.eval()
print(f"Loaded {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Q heads: {model.config.num_attention_heads}")
print(f"KV heads: {getattr(model.config, 'num_key_value_heads', model.config.num_attention_heads)}")
print(f"Max context: {model.config.max_position_embeddings}")

### 7.3 A real long-context prompt

This is ~1500 tokens of Wikipedia-style content about the Eiffel Tower. It has the structure we want:
- Named entities early (Anchor tier should save them)
- Lots of factual elaboration (Supporting tier material)
- A continuation the model should complete fluently

In [ ]:
LONG_PROMPT = """The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889. Locally nicknamed La dame de fer, it was constructed as the centerpiece of the 1889 World's Fair and to crown the centennial anniversary of the French Revolution. Although initially criticized by some of France's leading artists and intellectuals for its design, it has since become a global cultural icon of France and one of the most recognizable structures in the world. The Eiffel Tower is the most-visited paid monument in the world; 6.91 million people ascended it in 2015.

The tower is 330 meters tall, about the same height as an 81-story building, and the tallest structure in Paris. Its base is square, measuring 125 meters on each side. During its construction, the Eiffel Tower surpassed the Washington Monument to become the tallest man-made structure in the world, a title it held for 41 years until the Chrysler Building in New York City was finished in 1930. It was the first structure to reach a height of 300 meters. Due to the addition of a broadcasting aerial at the top of the tower in 1957, it is now taller than the Chrysler Building by 5.2 meters.

The tower has three levels for visitors, with restaurants on the first and second levels. The top level's upper platform is 276 meters above the ground, the highest observation deck accessible to the public in the European Union. Tickets can be purchased to ascend by stairs or elevator to the first and second levels. The climb from ground level to the first level is over 300 steps, as is the climb from the first level to the second. Although there is a staircase to the top level, it is usually accessible only by elevator.

Eiffel openly acknowledged that inspiration for the tower came from the Latting Observatory built in New York City in 1853. In May 1884, working at home, Maurice Koechlin, a senior engineer at the Compagnie des Etablissements Eiffel, made a sketch of their idea, described by him as a great pylon, consisting of four lattice girders standing apart at the base and coming together at the top, joined together by metal trusses at regular intervals.

Gustave Eiffel initially showed little enthusiasm for the project, but he did approve further study, and the two engineers then asked Stephen Sauvestre, the head of the company's architectural department, to contribute to the design. Sauvestre added decorative arches to the base of the tower, a glass pavilion to the first level, and other embellishments. Eiffel bought the rights to the patent on 13 September 1884. By 30 March 1885, Eiffel presented his plans to the Société des Ingénieurs Civils; after discussing the technical problems and emphasising the practical uses of the tower, he finished his talk by saying the tower would symbolise not only the art of the modern engineer, but also the century of industry and science in which we are living.

The proposed tower had been a subject of controversy, drawing criticism from those who did not believe it was feasible and those who objected on artistic grounds. These objections were an expression of a long-standing debate in France about the relationship between architecture and engineering. It came to a head as work began at the Champ de Mars: a Committee of Three Hundred led by the prominent architect Charles Garnier and including some of the most important figures of the arts, such as Adolphe Bouguereau, Guy de Maupassant, Charles Gounod and Jules Massenet, sent a petition to Jean-Charles Alphand, the Minister of Works and Commissioner for the Exhibition, and it was published by Le Temps on 14 February 1887.

Some of the protests had such remarkable foresight that they would prove nearly prophetic. Gustave Eiffel himself later wrote that the tower would endure because it embodies"""

# Show the token count
tokens = tok(LONG_PROMPT, return_tensors="pt")
print(f"Prompt token count: {tokens.input_ids.shape[-1]}")

### 7.4 Run the 2D eviction × quantization sweep

This is the headline experiment. It runs ~27 configurations (1 full + 2 uniform + 4 h2o + 4 streamingllm + 4×3 primekv) against the same long prompt and produces the Pareto scatter plot.

**Expected runtime: 5-15 min on T4, 2-5 min on A100.**

In [ ]:
%cd /content/PrimeKV
from primekv.sweep import sweep_2d_tradeoff, plot_report
import logging
logging.basicConfig(level=logging.WARNING)

# Extended prompt — multi-topic Wikipedia content, ~3500 tokens.
# Long enough to stress the cache, short enough to run in ~10 min on A100.
EXTENDED_PROMPT = LONG_PROMPT + """

The transformer architecture, introduced by Vaswani et al. in 2017, revolutionized sequence modeling by replacing recurrent neural networks with self-attention mechanisms. Unlike RNNs, which process tokens sequentially and struggle with long-range dependencies due to vanishing gradients, transformers attend to all positions simultaneously. This parallelism enables efficient training on GPUs and has been the foundation of every major language model since, including GPT, BERT, T5, and Llama. The self-attention mechanism computes three projections of each input token — a query, a key, and a value — and uses the query-key similarity to determine how much each token should attend to every other token. This results in quadratic complexity in sequence length, which is the primary bottleneck for long-context inference.

The KV cache is a critical optimization in transformer inference. During autoregressive generation, the model produces one token at a time, with each new token requiring attention over all previous tokens. Rather than recomputing the keys and values for prior tokens at every decode step, the model caches these projections from prefill. This transforms decode from quadratic to linear complexity per token, at the cost of substantial memory consumption. The cache size scales with layers, heads, head dimension, and sequence length; for large models with long contexts, it often exceeds the model weights in memory footprint.

Several techniques have been proposed to reduce KV cache memory. Quantization methods like KIVI and KVQuant compress the cached tensors from FP16 to INT8 or INT4, trading precision for memory. Eviction methods like H2O and StreamingLLM drop tokens deemed less important based on cumulative attention scores or positional heuristics. Prompt caching stores cache entries across requests for shared prefixes. Each approach exploits a different dimension of redundancy. Combining them — as PrimeKV does with its tiered storage policies — can achieve higher compression ratios than any single technique alone.

Paris, the capital of France, has been a major European city since its founding in the 3rd century BC by a Celtic tribe called the Parisii. Located on the Seine River in north-central France, Paris is renowned for its art, fashion, gastronomy, and culture. The city is home to iconic landmarks including the Louvre Museum, Notre-Dame Cathedral, the Arc de Triomphe, and the Eiffel Tower. With a metropolitan population of 12 million, Paris remains one of the most populous urban regions in Europe.

Returning to the engineer who gave his name to the tower: Gustave Eiffel was born in Dijon, France in 1832. He studied at the École Centrale des Arts et Manufactures in Paris, graduating in 1855. Before the tower that bears his name, Eiffel designed the internal iron framework of the Statue of Liberty in 1885, demonstrating his mastery of large-scale metallic structures. The Eiffel Tower was initially intended as a temporary structure for the 1889 World's Fair, scheduled to be dismantled after twenty years. Its value as a radio transmission tower during World War I saved it from destruction, and it has remained standing ever since. Eiffel died in Paris in 1923 at the age of 91.

The influence of the Eiffel Tower on subsequent architecture and engineering cannot be overstated. It demonstrated that iron could be used to construct buildings of unprecedented height, foreshadowing the skyscraper era. Moreover, the tower's visual design — its lattice structure, curved base arches, and tapering silhouette — influenced countless buildings throughout the 20th century. What makes the tower particularly fascinating is that"""

prompt_tokens = tok(EXTENDED_PROMPT, return_tensors='pt').input_ids.shape[-1]
print(f"Extended prompt length: {prompt_tokens} tokens\n")

# Aggressive capacities relative to ~3500 tokens means 1-15% retention.
# This is the regime where eviction methods actually have to make hard choices.
report = sweep_2d_tradeoff(
    model=model,
    tokenizer=tok,
    prompt=EXTENDED_PROMPT,
    eviction_caps=[32, 64, 128, 256, 512],
    precisions=["fp16", "int8", "int4"],
    decode_tokens=16,
    max_length=3500,
    device="cuda",
    progress=lambda msg: print(f"  {msg}"),
)

print(f"\nDone. {len(report.points)} configurations.")

### 7.5 Plot the results

In [ ]:
%cd /content/PrimeKV
import matplotlib.pyplot as plt

fig = plot_report(report, output_path="primekv_2d_sweep.png")
plt.show()

# Also save the raw numbers
with open("primekv_2d_sweep.csv", "w") as f:
    f.write(report.to_csv())
print("\nSaved: primekv_2d_sweep.png, primekv_2d_sweep.csv")

### 7.6 Headline numbers table

In [ ]:
%cd /content/PrimeKV
import pandas as pd

# Flatten the sweep results into a table.
rows = []
for p in report.points:
    rows.append({
        "cache": p.cache,
        "cap": p.extra.get("cap"),
        "precision": p.extra.get("precision"),
        "memory_MB": round(p.memory_bytes / (1024 * 1024), 2),
        "ratio": round(p.compression_ratio, 2),
        "ppl": round(p.perplexity, 3) if p.perplexity else None,
        "prefill_ms": round(p.prefill_ms, 1),
        "decode_ms": round(p.decode_ms, 1),
    })
df = pd.DataFrame(rows)
df = df.sort_values(["cache", "cap", "precision"])
print(df.to_string(index=False))

## 8. Honed sweeps: where does PrimeKV actually win?

The first round of results (Section 7) compared one-lever baselines (`uniform_int4`, pure `h2o`) against PrimeKV's two-lever design. That comparison is structurally unfair — int4 looked dominant on perplexity because no baseline was combining eviction *with* quantization. This section runs the sweeps that actually answer the question:

1. **8.1 Composed-baseline 2D sweep** — `h2o` and `streamingllm` now fill the full eviction × quantization plane via the new `H2OQuantCache` / `StreamingQuantCache` wrappers. PrimeKV no longer gets a free "only method with two levers" win.
2. **8.2 Long-context sweep** — capacity scales with prompt length (fixed *fraction* retained) so we test the regime where eviction has proportional leverage over fixed-ratio quantization.
3. **8.3 Seed aggregation** — same sweep with 5 sampling seeds, aggregated to mean/std. Without this, every number on a single chart is n=1.
4. **8.4 Reasoning-persistence harness** — perplexity averages across every token and hides whether the *right* tokens were kept. This runs small retention tests with a **filtered pass rate** (only count tests the `full` baseline passes), which was the bug in the earlier constraint-persistence charts.


### 8.1 2D sweep with composed baselines

Every capacity-driven method now fills the 2D surface. If PrimeKV still wins at the same compression ratio as `h2o_int4` / `streamingllm_int4`, that's real evidence the structural classifier is worth something. If not, we know where we are.


In [ ]:
%cd /content/PrimeKV
from primekv.sweep import sweep_2d_tradeoff, plot_report

report_2d = sweep_2d_tradeoff(
    model=model,
    tokenizer=tok,
    prompt=EXTENDED_PROMPT,
    eviction_caps=[32, 64, 128, 256, 512],
    precisions=["fp16", "int8", "int4"],
    decode_tokens=16,
    max_length=3500,
    device="cuda",
    progress=lambda msg: print(f"  {msg}"),
)
print(f"\nDone. {len(report_2d.points)} configurations.")
fig = plot_report(report_2d, output_path="primekv_2d_composed.png")
print("Saved: primekv_2d_composed.png")


### 8.2 Long-context sweep (capacity scales with length)

Unlike Section 7's fixed-length test, this sweeps length from 512 → 16k tokens with **capacity = 10% × length** (min 32). That is the regime where eviction methods should gain leverage over uniform int4: most tokens in a 16k context genuinely don't matter, but uniform quantization compresses every token the same amount regardless.

If PrimeKV doesn't separate from `uniform_int4` here, it probably doesn't separate anywhere.


In [ ]:
%cd /content/PrimeKV
from primekv.sweep import sweep_long_context, DEFAULT_LONG_LENGTHS, plot_report

# Qwen2.5-3B supports 32k context; trim the top of the default grid if
# your session has memory pressure.
long_lengths = [512, 1024, 2048, 4096, 8192]

report_lc = sweep_long_context(
    model=model,
    tokenizer=tok,
    prompt=EXTENDED_PROMPT,
    lengths=long_lengths,
    capacity_fraction=0.1,
    min_capacity=32,
    precisions=["int4"],
    caches=["uniform_int4", "h2o_int4", "streamingllm_int4", "primekv"],
    decode_tokens=16,
    device="cuda",
    progress=lambda msg: print(f"  {msg}"),
)
print(f"\nDone. {len(report_lc.points)} configurations.")
fig = plot_report(report_lc, output_path="primekv_long_context.png")
print("Saved: primekv_long_context.png")


### 8.3 Seed aggregation

With argmax decoding, a sweep is deterministic and n=1 is the same as n=100. To get honest error bars we switch to top-k sampling (`sample_top_k=50`) and run the same sweep across 5 seeds. `aggregate_reports` collapses the runs into mean/std per cell, so every chart plotted from it has real variance on it.


In [ ]:
%cd /content/PrimeKV
from primekv.sweep import sweep_pareto, aggregate_reports

seeds = [0, 1, 2, 3, 4]
reports = []
for s in seeds:
    r = sweep_pareto(
        model=model,
        tokenizer=tok,
        prompt=EXTENDED_PROMPT[:2000],  # shorter prompt keeps wall-time reasonable
        capacities=[32, 64, 128, 256],
        caches=["full", "uniform_int4", "h2o_int4", "streamingllm_int4", "primekv"],
        decode_tokens=16,
        max_length=1500,
        device="cuda",
        seed=s,
        sample_top_k=50,
        sample_temperature=1.0,
        progress=lambda msg, s=s: print(f"  seed={s} {msg}"),
    )
    reports.append(r)

agg = aggregate_reports(reports)
print(f"\nAggregated {len(reports)} runs into {len(agg.points)} cells.")
print("Each point now has perplexity_std, memory_bytes_std etc. in `extra`:")
for p in agg.points[:3]:
    extras = {k: v for k, v in p.extra.items() if k.endswith('_std') or k == 'n_runs'}
    print(f"  {p.cache} cap={p.sweep_value}  ppl={p.perplexity:.3f}  {extras}")


### 8.4 Reasoning-persistence harness (filtered pass rate)

The earlier constraint-persistence chart had `primekv_spacy` passing 100% of tests while `full` passed only 50% — which is a physical impossibility for cache quality. The cause: the denominator included tests where `full` was already failing for non-cache reasons (model couldn't comply with the constraint, couldn't chain the facts), so any perturbation of generation could flip those tests and look like a win.

`run_reasoning_suite` + `filtered_pass_rate` fix this: only count tests where `full` itself passes. That's the honest cache-quality metric — among questions the uncompressed model can actually answer, how many survive under each cache?

The built-in suite is a small smoke test. Swap in LongBench / RULER for a real evaluation.


In [ ]:
%cd /content/PrimeKV
from primekv.reasoning import (
    default_reasoning_suite,
    run_reasoning_suite,
    plot_reasoning_report,
)
from primekv.baselines import (
    FullCache,
    H2OQuantCache,
    StreamingQuantCache,
    UniformQuantCache,
)
from primekv.cache import PrimeKVCache
from primekv.classifier import RuleBasedClassifier, Tier

n_layers = model.config.num_hidden_layers

# Factories so each (test, seed, cache) run gets a fresh cache — matters
# for PrimeKV because reset() preserves the classifier assignment.
factories = {
    "full":               lambda: FullCache(n_layers),
    "uniform_int4":       lambda: UniformQuantCache(n_layers, bits=4),
    "h2o_int4":           lambda: H2OQuantCache(n_layers, capacity=64, bits=4),
    "streamingllm_int4":  lambda: StreamingQuantCache(n_layers, num_sinks=4, window=64, bits=4),
    "primekv":            lambda: PrimeKVCache(
        n_layers,
        classifier=RuleBasedClassifier(anchor_prefix_len=8, semantic_stride=3),
        max_entries_per_tier={Tier.SUPPORTING: 64},
    ),
}
caches = {name: f() for name, f in factories.items()}

tests = default_reasoning_suite()
seeds = [0, 1, 2]  # tiny: sampling noise, not a full statistical test

report_rs = run_reasoning_suite(
    caches=caches,
    tests=tests,
    model=model,
    tokenizer=tok,
    seeds=seeds,
    device="cuda",
    sample_top_k=40,
    sample_temperature=0.8,
    cache_factories=factories,
    progress=lambda msg: print(f"  {msg}"),
)

print("\nSummary:")
for cache_name, row in report_rs.summary().items():
    print(
        f"  {cache_name:20s}  filtered={row['filtered_pass_rate']:.2%}  "
        f"raw={row['raw_pass_rate']:.2%}  "
        f"(n_valid_tests={row['n_valid_tests']})"
    )

fig = plot_reasoning_report(report_rs, output_path="primekv_reasoning.png")
print("\nSaved: primekv_reasoning.png")


### 8.5 What to look for in the plots

Across Sections 8.1–8.4, three specific questions are worth checking:

- **2D plot (8.1):** does the PrimeKV cloud have any point that strictly dominates the `h2o_int4` / `streamingllm_int4` points at equal compression ratio? That's the real PrimeKV-wins-on-quality signal. If all the composed-baseline points sit on or below PrimeKV's frontier, the structural classifier isn't buying anything that eviction+quant alone doesn't already buy.
- **Long-context (8.2):** at 8k/16k tokens, does PrimeKV pull away from `uniform_int4`? Uniform quantization is a fixed 4× compression regardless of length; eviction-at-fixed-fraction compresses proportionally to length, so the two should separate as context grows. If they don't, the structural claim is weak.
- **Reasoning (8.4):** on the *filtered* pass rate (not the raw), does PrimeKV beat `h2o_int4`? This is where structure *should* matter — tests that require recalling specific tokens punish eviction that drops those tokens, regardless of how precisely the survivors are stored.

Any one of these three being positive is a real wedge. All three being flat means the method doesn't have a regime, and we should know that now rather than later.
